<a href="https://colab.research.google.com/github/LLoghan/analise-preco-jogos/blob/main/Preco_Jogos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Carregamento dos dados e limpeza
df = pd.read_csv('Preco_Jogos.csv')
df.columns = df.columns.str.strip()

colunas_preco = [
    'Preço Steam', 'Preço Playstation', 'Preço Xbox',
    'Preço Nintendo', 'Preço EpicGames'
]

def limpar_preco(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip()
    if 'não disponível' in texto.lower() or texto == '':
        return np.nan
    texto = re.sub(r'\(.*?\)', '', texto)
    texto = texto.replace('R$', '').strip()
    if ',' in texto and '.' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    elif ',' in texto:
        texto = texto.replace(',', '.')
    try:
        return float(texto)
    except ValueError:
        return np.nan

for col in colunas_preco:
    df[col] = df[col].apply(limpar_preco)

mapa_nomes = {
    'Preço Steam': 'Steam',
    'Preço Playstation': 'PlayStation',
    'Preço Xbox': 'Xbox',
    'Preço Nintendo': 'Nintendo',
    'Preço EpicGames': 'Epic Games'
}
df_plot = df.rename(columns=mapa_nomes)
plataformas = list(mapa_nomes.values())

# Estatísticas em PT-BR no terminal
estatisticas = df_plot[plataformas].agg(['count', 'mean', 'median', 'min', 'max', 'std']).T
estatisticas.columns = ['Contagem', 'Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão']
print("=== Estatísticas Descritivas por Plataforma (PT-BR) ===")
print(estatisticas.round(2))


# --- GRÁFICO 1: Pizza  ---
custo_beneficio = (df_plot[plataformas].count() / df_plot[plataformas].mean()) * 100

plt.figure(figsize=(8, 6))
plt.gcf().patch.set_facecolor('white')
cores_pizza = ['#1b2838', '#003791', '#107c10', '#e60012', '#555555']

wedges, texts, autotexts = plt.pie(
    custo_beneficio,
    labels=custo_beneficio.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=cores_pizza,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)

for text in texts:
    text.set_fontweight('bold')
    text.set_fontsize(11)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

plt.title('Custo-Benefício por Plataforma\n(Relação: Catálogo Disponível vs. Preço Médio)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('grafico_custo_beneficio.png', dpi=300, bbox_inches='tight')
plt.show()


# --- GRÁFICO 2: Torres ---
df_plot['Max_Val'] = df_plot[plataformas].max(axis=1)
df_plot['Min_Val'] = df_plot[plataformas].min(axis=1)

top_exp = df_plot.sort_values(by='Max_Val', ascending=False).head(5)
top_cheap = df_plot[df_plot['Min_Val'] > 0].sort_values(by='Min_Val', ascending=True).head(5)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

cores_caros_vividas = ['#FF1E56', '#FF5722', '#FF9800', '#E91E63', '#D500F9']
cores_baratos_vividas = ['#00E676', '#00B0FF', '#1DE9B6', '#00E5FF', '#76FF03']

bars1 = axes[0].bar(top_exp['Jogo'], top_exp['Max_Val'], color=cores_caros_vividas, edgecolor='black', linewidth=1.2, width=0.55)
axes[0].set_title('Top 5 Jogos Mais Caros (Torres)', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Preço Máximo (R$)', fontweight='bold')
axes[0].set_ylim(0, 410)
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars1:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, yval + 6, f'R$ {yval:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9.5)

bars2 = axes[1].bar(top_cheap['Jogo'], top_cheap['Min_Val'], color=cores_baratos_vividas, edgecolor='black', linewidth=1.2, width=0.55)
axes[1].set_title('Top 5 Jogos Mais Baratos (Torres)', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Preço Mínimo (R$)', fontweight='bold')
axes[1].set_ylim(0, 35)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars2:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.6, f'R$ {yval:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9.5)

plt.tight_layout()
plt.savefig('graficos_torres_vividos.png', dpi=300, bbox_inches='tight')
plt.show()


# --- GRÁFICO 3: Circulos ---
plt.figure(figsize=(9, 6))
plt.gcf().patch.set_facecolor('white')
media_plataformas = df_plot[plataformas].mean().round(2)

cores_conjuntos_vividas = ['#00B4D8', '#3A86FF', '#38B000', '#FF0054', '#FFBE0B']

plt.scatter(
    media_plataformas.index,
    media_plataformas.values,
    s=media_plataformas.values * 18,
    color=cores_conjuntos_vividas,
    alpha=0.9,
    edgecolors="black",
    linewidth=2
)

for i, txt in enumerate(media_plataformas.values):
    plt.annotate(
        f"Média: R$ {txt:.2f}",
        (i, media_plataformas.values[i]),
        textcoords="offset points",
        xytext=(0, 34),
        ha='center',
        fontweight='bold'
    )

plt.title('Conjunto e Comparação dos Valores Médios por Plataforma', fontweight='bold', fontsize=13)
plt.xlabel('Plataforma', fontweight='bold')
plt.ylabel('Preço Médio (R$)', fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.ylim(130, 225)
plt.tight_layout()
plt.savefig('grafico_conjuntos_vivido.png', dpi=300, bbox_inches='tight')
plt.show()